In [1]:
import os
import joblib
import pandas as pd
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from Preprocessing_Pipeline import preprocess_arabic , normalize_arabic

c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
[2026-08-22 15:13:46,076 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [2]:
df = pd.read_csv("../Datasets/Final_Data.csv")
Arab_df = df.copy()
Arab_df.head(10)

,review_description,rating,company
0,رائع,positive,talbat
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,positive,talbat
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,negative,talbat
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,talbat
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,negative,talbat
5,أصبح غالي جداً,negative,talbat
6,جميل جدا رائع. . .,positive,talbat
7,للأسف الواحد ينصدم بعد زيادة الاسعار و للاسف ب...,negative,talbat
8,برنامج توترز توصيل احلى من برنامجكم فاشل,negative,talbat
9,كتير في تحسن خدمة العملاء لطفين في بعض الاخطاء...,positive,talbat


In [3]:
Arab_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40046 entries, 0 to 40045
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   review_description  40045 non-null  str  
 1   rating              40046 non-null  str  
 2   company             40046 non-null  str  
dtypes: str(3)
memory usage: 5.0 MB


In [4]:
Arab_df['review_description'].duplicated().sum()

1042

In [5]:
Arab_df.drop_duplicates(subset=['review_description'], keep='first', inplace=True)

In [6]:
Arab_df.drop(columns='company',inplace=True)

In [7]:
def map_sentiment(val):
    val = str(val).strip().lower()
    if val in ['positive', '5', '4', 'ممتاز', 'إيجابي']:
        return 'Positive'
    elif val in ['negative', '1', '2', 'سيء', 'سلبي']:
        return 'Negative'
    return 'Neutral'

Arab_df['sentiment'] = Arab_df['rating'].apply(map_sentiment)

In [8]:
X = Arab_df['review_description']

y = Arab_df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
X_train_clean = X_train.apply(normalize_arabic)

X_test_clean = X_test.apply( normalize_arabic)

In [10]:
print("\nCreating TF-IDF features...")

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True
)

X_train_tf = char_vectorizer.fit_transform(X_train_clean)

X_test_tf = char_vectorizer.transform(X_test_clean)


Creating TF-IDF features...


In [11]:
# model = LinearSVC(class_weight='balanced', max_iter=5000)

# model.fit(X_train_tf,y_train)
# y_pred = model.predict(  X_test_tf)
models = {
    "MultinomialNB": MultinomialNB(),
    "LinearSVC": LinearSVC(random_state=42),
    "LogisticRegression": LogisticRegression( max_iter=1000, random_state=42 ),
}

results = []
fitted_models = {}

for name, model in models.items():
    start_train = time.time()
    model.fit(X_train_tf, y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    val_preds = model.predict(X_test_tf)
    predict_time = time.time() - start_pred

    acc = accuracy_score(y_test, val_preds)
  

    fitted_models[name] = model
    results.append({
        "model": name,
        "val_accuracy": acc,
        "train_time_sec": train_time,
        "predict_time_sec": predict_time,
    })

result = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
result

,model,val_accuracy,train_time_sec,predict_time_sec
2,LogisticRegression,0.843994,37.069548,0.015585
1,LinearSVC,0.838354,4.127150,0.043001
0,MultinomialNB,0.832842,0.303494,0.021008


In [12]:
best_model_name = result.iloc[0]["model"]
best_model = fitted_models[best_model_name]

In [13]:
y_pred = best_model.predict( X_test_tf)

In [14]:
accuracy = accuracy_score( y_test,val_preds)
print("\n" + "=" * 60)
print("Arabic SENTIMENT MODEL RESULTS")
print("=" * 60)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")

print( classification_report(  y_test,y_pred))

print("\nConfusion Matrix:")

print( confusion_matrix(y_test, y_pred))


Arabic SENTIMENT MODEL RESULTS

Accuracy: 0.8440

Classification Report:
              precision    recall  f1-score   support

    Negative       0.83      0.80      0.82      2787
     Neutral       0.38      0.01      0.02       380
    Positive       0.85      0.94      0.89      4634

    accuracy                           0.84      7801
   macro avg       0.69      0.58      0.57      7801
weighted avg       0.82      0.84      0.82      7801


Confusion Matrix:
[[2237    1  549]
 [ 165    3  212]
 [ 286    4 4344]]


In [15]:

joblib.dump(best_model, 'Arabic_model_Weights.pkl')
joblib.dump(char_vectorizer, 'Arabic_model_Vectorizer.pkl')
print("Best model saved to Arabic_model_Weights.pkl")

Best model saved to Arabic_model_Weights.pkl
